In [83]:
import numpy as np
import pandas as pd

In [84]:
df = pd.read_csv("../Data/trainTitanic2.csv")

In [85]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [87]:
df.drop(columns = ['Cabin'] , inplace=True)

In [88]:
df.dropna(subset=['Embarked'], inplace=True)

In [89]:
df['Age'].fillna(df['Age'].mean() , inplace=True)

C:\Users\gdeep\AppData\Local\Temp\ipykernel_3600\3516665944.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].mean() , inplace=True)


In [90]:
X = df[['Age' , 'Fare' , 'SibSp' , 'Parch']].values
y = df['Survived'].values

In [91]:
from sklearn.model_selection import train_test_split
X_train, X_test ,y_train ,y_test = train_test_split(X, y , test_size=0.2 , stratify=y)

## From scratch

In [92]:
class GaussianNB:
    def __init__(self):
        self.classes = None
        self.priors = None
        self.means = None
        self.variances = None
        
    def fit(self , X ,y):
        self.classes = np.unique(y)
        number_classes = len(self.classes)
        number_features = X.shape[1]
        
        self.priors = np.zeros(number_classes)
        self.means = np.zeros((number_classes , number_features))
        self.variances = np.zeros((number_classes , number_features))
        
        for i , c in enumerate(self.classes):
            X_Class = X[y==c] # only choose of that class , and calc mean variances for the features
            self.priors[i] = len(X_Class)/len(X)
            self.means[i] = np.mean(X_Class, axis=0)
            self.variances[i] = np.var(X_Class , axis=0)
            
    
    def normal_prob(self , X , mean , variance):
        numerator = np.exp(-((X - mean) ** 2) / (2 * variance))
        denominator = np.sqrt(2 * np.pi * variance)
        return numerator/denominator
    
    def predict(self , X):
        predictions = []
        for x in X:
            class_probabilites = []
            for i , c in enumerate(self.classes):
                probability = self.priors[i]
                for j in range(X.shape[1]):
                    probability *= self.normal_prob(x[j] , self.means[i,j] , self.variances[i,j] )
                class_probabilites.append(probability)
                
            best_class = self.classes[np.argmax(class_probabilites)]
            predictions.append(best_class)
        return np.array(predictions)
    
    #too many features , and probabilites will go towards 0 , log can be used in place of multiplication , may be later

In [93]:
normal_nb = GaussianNB()
normal_nb.fit(X_train , y_train)
y_pred = normal_nb.predict(X_test)

In [96]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.6404494382022472

In [97]:
from sklearn.naive_bayes import GaussianNB

gnb_sklearn = GaussianNB()
gnb_sklearn.fit(X_train, y_train)
y_pred_sklearn = gnb_sklearn.predict(X_test)
accuracy_score(y_test, y_pred_sklearn)

0.6404494382022472

In [99]:
print("Scratch means:")
print(normal_nb.means)

print("Sklearn means:")
print(gnb_sklearn.theta_)

Scratch means:
[[30.22209295 21.222559    0.53758542  0.30296128]
 [29.02653637 50.44955551  0.5         0.45955882]]
Sklearn means:
[[30.22209295 21.222559    0.53758542  0.30296128]
 [29.02653637 50.44955551  0.5         0.45955882]]


## DONE AND DUSTED